### Evaluating, base and reasoning models

* downloading the model here

In [8]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent  # Get parent of current directory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from downloading_the_base_model.download_model import downlaod_model

downlaod_model("Qwen/Qwen3-0.6B", "qwen")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

'/teamspace/studios/this_studio/open-posttraining-system/evaluating_reasoning_models/qwen'

* ok, so we are loading the model and tokenizer here,

In [9]:


import torch
from safetensors.torch import load_file

from base_model.qwen import (
    QWEN_CONFIG_06_B,
    Qwen3Model,
    Qwen3Tokenizer,
    generate_text,
    load_hf_weights_into_qwen,
)

model_dir = Path.cwd() / "qwen"

#def main(prompt):
def load_model_and_tokenizer(which_model, use_compile):

    if which_model == "base":
        tokenizer = Qwen3Tokenizer(model_dir / "tokenizer.json")
        
    elif which_model == "reasoning":
        tokenizer = Qwen3Tokenizer(model_dir / "tokenizer.json", apply_chat_template=True,   add_generation_prompt=True, add_thinking=True)
    
    

    else:
        raise ValueError(f"Not a valid model type")
    
    weights = load_file(model_dir / "model.safetensors")

    model = Qwen3Model(QWEN_CONFIG_06_B)
    load_hf_weights_into_qwen(
        model,
        param_config={
            "n_layers": QWEN_CONFIG_06_B["n_layers"],
            "hidden_dim": QWEN_CONFIG_06_B["hidden_dim"],
        },
        params=weights,
    )
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.to(torch.bfloat16)

    if use_compile:
        torch._dynamo.config.allow_unspec_int_on_nn_module = True
        model = torch.compile(model)
     
    return model, tokenizer




In [10]:
### we are using the --> "base model"
which_model = "base"

model, tokenizer = load_model_and_tokenizer(which_model=which_model, use_compile=False)

* here, we will be using kv cache---> to generate text

In [12]:
import torch
from base_model.qwen import KVCache

device = "cuda" if torch.cuda.is_available() else "cpu"

@torch.inference_mode()

def generate_text_stream_with_kv_cache(prompt, model, tokenizer, device, max_new_tokens, eos_token_id):
    # encode prompt and move to device
    input_ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)

    ## setting the model --> to evaluation mode
    model.eval()

    ## initialize the KV Cache
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    ## initial forward pass using full prompt -->  then will  cache this much
    logits = model(input_ids, cache=cache)[:, -1]

    for _ in range(max_new_tokens):

        ## greedy decoding --> cause we are not going to pass--> these into softmax..later
        next_token = torch.argmax(logits, dim=-1, keepdim=True)

        ## stop if EOS token is generated
        if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
            break

        ## compute -> store single token in memory -> pass it forward -->  and forget...  (used for streaming tokens) --(memory efficient)
        yield next_token

        ## ok, now this pass uses only new token---> cause rest of the previous tokens  are cache..
        logits = model(next_token, cache=cache)[:, -1]


def generate_text(prompt, model, tokenizer, device, max_new_tokens, eos_token_id):

    ## remember this loop is calling generator object--> it  will receive tokens till the generator stops yielding
    ## like: --> next(generator)
    for token in generate_text_stream_with_kv_cache(
        prompt=prompt,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        eos_token_id=eos_token_id,
        device=device,
    ):

        output_tokens = token.squeeze(0).tolist() ## squeeze and later converts tensor -> list

        print(tokenizer.decode(output_tokens), end="", flush=True)




In [13]:
tokenizer.eos_token_id

151645

In [17]:
max_new_tokens = 50
device = device
eos_token_id = tokenizer.eos_token_id
prompt = ("If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?")

## calling text-generation function
generate_text(prompt, model, tokenizer, device, max_new_tokens, eos_token_id)


 To solve this problem, we can use

 the identity that relates the sum and product of two numbers to their squares. The identity is:

$$
a^2 + b^2 = (a + b)^2 - 2ab
$$



* max_token_reached --> that is why it stopped.

* visualizing the terrible output into --> clean math

In [19]:
from IPython.display import Math
display(Math(r"$$a^2 + b^2 = (a + b)^2 - 2ab$$"))

<IPython.core.display.Math object>

### creating a wrapper function